# Exercise 2.4: GCN and GraphSAGE on the Warsaw Bike-Sharing Graph

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_4_warsaw_gcn_graphsage_spatial_baselines.ipynb)

This notebook compares feature-only models with graph neural networks on bike-sharing station demand.

- Data source: Warsaw Bike-Sharing Daily Periods Graph Dataset for GNN, Season 2023, Mendeley Data, DOI: 10.17632/kzvdgfzk4w.1.
- Task: predict station-level trip intensity from station, weather, time, and spatial graph context.
- Models: Random Forest, XGBoost, GCN, GraphSAGE, and GraphSAGE with shuffled edges.
- Main question: when does the graph add useful spatial information beyond ordinary tabular features?

## 1. Setup

The GNN models are implemented directly in PyTorch. This keeps the notebook light and avoids a PyTorch Geometric installation step.

In [ ]:
# Colab setup. In a local environment, run this only if a package is missing.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install networkx scikit-learn xgboost matplotlib pandas gdown

In [ ]:
from pathlib import Path
import math
import pickle
import random
import re
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 2. Load a Warsaw graph

Use one real `.pt` graph file from the Warsaw dataset. The Mendeley file can be blocked in Colab, so this notebook is set up for a Google Drive copy shared by the course instructor.

The default file is `afternoon_peak_01_06_2023.pt`, the example file named in the dataset description. It is about 1.3 MB and contains 295 stations and 1684 directed flow edges.

Instructor step: download this file once, upload it to Google Drive, set access to anyone with the link, and paste the share link or file id below.

In [ ]:
GRAPH_FILENAME = "afternoon_peak_01_06_2023.pt"
GRAPH_FILE = Path("/content") / GRAPH_FILENAME if IN_COLAB else Path("data") / GRAPH_FILENAME

# Paste either the full Google Drive sharing URL or only the file id.
GOOGLE_DRIVE_SHARE_URL = ""
GOOGLE_DRIVE_FILE_ID = ""

def extract_google_drive_file_id(url: str) -> str | None:
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None

def google_drive_source() -> str:
    if GOOGLE_DRIVE_FILE_ID.strip():
        return f"https://drive.google.com/uc?id={GOOGLE_DRIVE_FILE_ID.strip()}"
    if GOOGLE_DRIVE_SHARE_URL.strip():
        file_id = extract_google_drive_file_id(GOOGLE_DRIVE_SHARE_URL.strip())
        if file_id:
            return f"https://drive.google.com/uc?id={file_id}"
        return GOOGLE_DRIVE_SHARE_URL.strip()
    raise ValueError(
        "No Google Drive source configured. Upload afternoon_peak_01_06_2023.pt to Google Drive, "
        "share it with anyone who has the link, then paste the link into GOOGLE_DRIVE_SHARE_URL "
        "or paste the file id into GOOGLE_DRIVE_FILE_ID."
    )

def download_warsaw_graph(graph_file: Path) -> Path:
    if graph_file.exists():
        print(f"Using cached graph: {graph_file}")
        return graph_file

    try:
        import gdown
    except ImportError as exc:
        raise ImportError("Install gdown first: pip install gdown") from exc

    graph_file.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {GRAPH_FILENAME} from Google Drive...")
    downloaded = gdown.download(google_drive_source(), str(graph_file), quiet=False, fuzzy=True)
    if downloaded is None or not graph_file.exists() or graph_file.stat().st_size == 0:
        raise RuntimeError("Google Drive download failed. Check that the file is shared with anyone who has the link.")
    return graph_file

def load_graph(graph_file: Path) -> nx.DiGraph:
    graph_path = download_warsaw_graph(graph_file)
    with graph_path.open("rb") as f:
        graph = pickle.load(f)
    if not isinstance(graph, nx.Graph):
        raise TypeError(f"Expected a NetworkX graph, got {type(graph)!r}")
    return nx.DiGraph(graph)

G = load_graph(GRAPH_FILE)
print(f"nodes={G.number_of_nodes():,}, edges={G.number_of_edges():,}, file={GRAPH_FILE}")

## 3. Convert graph attributes into a node table

The target is station-level trip intensity: incoming plus outgoing `trips_count`, transformed with `log1p`. The feature-only models see only node attributes. The GNNs see the same node attributes plus the graph edges.

In [ ]:
def edge_weight(edge_data: dict) -> float:
    for key in ["trips_count", "trip_count", "count", "weight"]:
        if key in edge_data:
            try:
                return float(edge_data[key])
            except Exception:
                return 0.0
    return 1.0

def graph_to_node_frame(graph: nx.DiGraph) -> pd.DataFrame:
    total_flow = {node: 0.0 for node in graph.nodes}
    in_flow = {node: 0.0 for node in graph.nodes}
    out_flow = {node: 0.0 for node in graph.nodes}
    for u, v, data in graph.edges(data=True):
        w = edge_weight(data)
        out_flow[u] += w
        in_flow[v] += w
        total_flow[u] += w
        total_flow[v] += w

    rows = []
    for node, attrs in graph.nodes(data=True):
        row = {"node_id": node, **dict(attrs)}
        row["target_total_trips"] = total_flow[node]
        row["target_in_trips"] = in_flow[node]
        row["target_out_trips"] = out_flow[node]
        rows.append(row)
    return pd.DataFrame(rows)

nodes = graph_to_node_frame(G)
nodes["y_log_total_trips"] = np.log1p(nodes["target_total_trips"].astype(float))

exclude_exact = {"node_id", "target_total_trips", "target_in_trips", "target_out_trips", "y_log_total_trips"}
exclude_contains = ["trip", "flow", "target"]
candidate_features = []
for column in nodes.columns:
    if column in exclude_exact or any(term in column.lower() for term in exclude_contains):
        continue
    numeric = pd.to_numeric(nodes[column], errors="coerce")
    if numeric.notna().sum() >= max(5, int(0.2 * len(nodes))) and numeric.nunique(dropna=True) > 1:
        nodes[column] = numeric
        candidate_features.append(column)

if not candidate_features:
    raise ValueError("No numeric node features were found. Inspect the node attributes and select features manually.")

feature_frame = nodes[candidate_features].copy()
feature_frame = feature_frame.replace([np.inf, -np.inf], np.nan)
feature_frame = feature_frame.fillna(feature_frame.median(numeric_only=True)).fillna(0)

print(f"Using {len(candidate_features)} node features:")
print(candidate_features)
nodes[["node_id", "target_total_trips", "y_log_total_trips"] + candidate_features[:8]].head()

In [ ]:
edges = pd.DataFrame(
    [{"source": u, "target": v, "trips_count": edge_weight(data)} for u, v, data in G.edges(data=True)]
)
display(edges.head())

fig, ax = plt.subplots(figsize=(7, 4))
nodes["target_total_trips"].hist(ax=ax, bins=30, color="#3f7f93", edgecolor="white")
ax.set_title("Station trip intensity distribution")
ax.set_xlabel("incoming + outgoing trips")
ax.set_ylabel("stations")
plt.show()

## 4. Train/test split and feature-only baselines

Random Forest and XGBoost use the same node table but no graph edges. This is the ordinary tabular baseline.

In [ ]:
X = feature_frame.to_numpy(dtype=np.float32)
y = nodes["y_log_total_trips"].to_numpy(dtype=np.float32)

train_idx, test_idx = train_test_split(
    np.arange(len(nodes)), test_size=0.30, random_state=SEED
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "rmse": float(math.sqrt(mse)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

results = []
predictions = {}

rf = RandomForestRegressor(n_estimators=350, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
rf.fit(X_scaled[train_idx], y[train_idx])
pred_rf = rf.predict(X_scaled[test_idx])
results.append({"model": "Random Forest", **regression_metrics(y[test_idx], pred_rf)})
predictions["Random Forest"] = pred_rf

try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.035,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=SEED,
    )
    xgb.fit(X_scaled[train_idx], y[train_idx])
    pred_xgb = xgb.predict(X_scaled[test_idx])
    results.append({"model": "XGBoost", **regression_metrics(y[test_idx], pred_xgb)})
    predictions["XGBoost"] = pred_xgb
except Exception as exc:
    print(f"XGBoost unavailable ({exc}). Using sklearn HistGradientBoostingRegressor as a fallback.")
    from sklearn.ensemble import HistGradientBoostingRegressor

    hgb = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.035, random_state=SEED)
    hgb.fit(X_scaled[train_idx], y[train_idx])
    pred_hgb = hgb.predict(X_scaled[test_idx])
    results.append({"model": "HistGradientBoosting", **regression_metrics(y[test_idx], pred_hgb)})
    predictions["HistGradientBoosting"] = pred_hgb

pd.DataFrame(results).sort_values("rmse")

## 5. Build graph tensors

The GNNs receive an adjacency matrix from the station graph. We also create a shuffled-edge graph as a negative control. If the real graph is useful, the real-edge GNN should beat the shuffled-edge version.

In [ ]:
node_order = list(nodes["node_id"])
node_to_pos = {node: i for i, node in enumerate(node_order)}

edge_pairs = []
for u, v in G.edges():
    if u in node_to_pos and v in node_to_pos:
        edge_pairs.append((node_to_pos[u], node_to_pos[v]))

if not edge_pairs:
    raise ValueError("The graph has no usable edges after node alignment.")

def make_undirected_edges(edge_pairs: list[tuple[int, int]]) -> list[tuple[int, int]]:
    undirected = set()
    for u, v in edge_pairs:
        if u == v:
            continue
        undirected.add((u, v))
        undirected.add((v, u))
    return sorted(undirected)

def shuffled_edges(edge_pairs: list[tuple[int, int]], n_nodes: int, seed: int = SEED) -> list[tuple[int, int]]:
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_nodes)
    shuffled = [(int(perm[u]), int(perm[v])) for u, v in edge_pairs if perm[u] != perm[v]]
    return shuffled

def row_normalized_adjacency(edge_pairs: list[tuple[int, int]], n_nodes: int, add_self: bool) -> torch.Tensor:
    A = torch.zeros((n_nodes, n_nodes), dtype=torch.float32)
    for u, v in edge_pairs:
        A[v, u] = 1.0
    if add_self:
        A += torch.eye(n_nodes, dtype=torch.float32)
    degree = A.sum(dim=1, keepdim=True).clamp(min=1.0)
    return A / degree

n = len(nodes)
real_edges = make_undirected_edges(edge_pairs)
random_edges = make_undirected_edges(shuffled_edges(edge_pairs, n))
adj_mean = row_normalized_adjacency(real_edges, n, add_self=False).to(DEVICE)
adj_gcn = row_normalized_adjacency(real_edges, n, add_self=True).to(DEVICE)
adj_mean_shuffled = row_normalized_adjacency(random_edges, n, add_self=False).to(DEVICE)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
y_tensor = torch.tensor(y.reshape(-1, 1), dtype=torch.float32, device=DEVICE)
train_tensor = torch.tensor(train_idx, dtype=torch.long, device=DEVICE)
test_tensor = torch.tensor(test_idx, dtype=torch.long, device=DEVICE)

print(f"real undirected edges={len(real_edges):,}, shuffled undirected edges={len(random_edges):,}")

## 6. GCN and GraphSAGE in PyTorch

- GCN smooths node representations with the normalized adjacency matrix.
- GraphSAGE concatenates each station's own features with the mean feature vector from neighboring stations.
- The shuffled-edge GraphSAGE model has the same node features and a graph of similar size, but the spatial relations are wrong.

In [ ]:
class GCNRegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_dim, hidden_dim)
        self.lin2 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)
        self.dropout = torch.nn.Dropout(0.15)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h = torch.relu(self.lin1(adj @ x))
        h = self.dropout(h)
        h = torch.relu(self.lin2(adj @ h))
        return self.out(h)

class GraphSAGELayer(torch.nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin = torch.nn.Linear(in_dim * 2, out_dim)

    def forward(self, x: torch.Tensor, adj_mean: torch.Tensor) -> torch.Tensor:
        neighbor_mean = adj_mean @ x
        h = torch.cat([x, neighbor_mean], dim=1)
        return torch.relu(self.lin(h))

class GraphSAGERegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.sage1 = GraphSAGELayer(in_dim, hidden_dim)
        self.sage2 = GraphSAGELayer(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)
        self.dropout = torch.nn.Dropout(0.15)

    def forward(self, x: torch.Tensor, adj_mean: torch.Tensor) -> torch.Tensor:
        h = self.sage1(x, adj_mean)
        h = self.dropout(h)
        h = self.sage2(h, adj_mean)
        return self.out(h)

def train_gnn(model: torch.nn.Module, adj: torch.Tensor, epochs: int = 500, lr: float = 0.01) -> tuple[np.ndarray, list[float]]:
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = torch.nn.MSELoss()
    history = []
    best_state = None
    best_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_tensor, adj)
        loss = loss_fn(pred[train_tensor], y_tensor[train_tensor])
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
        if history[-1] < best_loss:
            best_loss = history[-1]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_pred = model(X_tensor, adj)[test_tensor].detach().cpu().numpy().ravel()
    return test_pred, history

torch.manual_seed(SEED)
pred_gcn, hist_gcn = train_gnn(GCNRegressor(X_scaled.shape[1]), adj_gcn)
results.append({"model": "GCN", **regression_metrics(y[test_idx], pred_gcn)})
predictions["GCN"] = pred_gcn

torch.manual_seed(SEED)
pred_sage, hist_sage = train_gnn(GraphSAGERegressor(X_scaled.shape[1]), adj_mean)
results.append({"model": "GraphSAGE", **regression_metrics(y[test_idx], pred_sage)})
predictions["GraphSAGE"] = pred_sage

torch.manual_seed(SEED)
pred_sage_shuffled, hist_sage_shuffled = train_gnn(GraphSAGERegressor(X_scaled.shape[1]), adj_mean_shuffled)
results.append({"model": "GraphSAGE shuffled edges", **regression_metrics(y[test_idx], pred_sage_shuffled)})
predictions["GraphSAGE shuffled edges"] = pred_sage_shuffled

pd.DataFrame(results).sort_values("rmse")

## 7. Compare results

Lower RMSE and MAE are better. Higher R2 is better.

In [ ]:
metrics_table = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(metrics_table)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, color in zip(axes, ["rmse", "mae", "r2"], ["#507dbc", "#3f7f93", "#8a6f3d"]):
    ordered = metrics_table.sort_values(metric, ascending=(metric != "r2"))
    ax.barh(ordered["model"], ordered[metric], color=color)
    ax.set_title(metric.upper())
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
best_model = metrics_table.iloc[0]["model"]
best_pred = predictions[best_model]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y[test_idx], best_pred, alpha=0.75, color="#2f6f73", edgecolor="white", linewidth=0.5)
lims = [min(y[test_idx].min(), best_pred.min()), max(y[test_idx].max(), best_pred.max())]
ax.plot(lims, lims, color="#333333", linestyle="--", linewidth=1)
ax.set_xlabel("observed log1p trips")
ax.set_ylabel("predicted log1p trips")
ax.set_title(f"Observed vs predicted: {best_model}")
ax.grid(alpha=0.25)
plt.show()

## 8. Spatial view

If latitude and longitude are available, plot stations by target intensity. Spatial clusters are one reason graph models can help.

In [ ]:
lat_candidates = [c for c in nodes.columns if c.lower() in {"lat", "latitude", "y"}]
lng_candidates = [c for c in nodes.columns if c.lower() in {"lng", "lon", "longitude", "x"}]

if lat_candidates and lng_candidates:
    lat_col, lng_col = lat_candidates[0], lng_candidates[0]
    plot_nodes = nodes.copy()
    plot_nodes[lat_col] = pd.to_numeric(plot_nodes[lat_col], errors="coerce")
    plot_nodes[lng_col] = pd.to_numeric(plot_nodes[lng_col], errors="coerce")
    plot_nodes = plot_nodes.dropna(subset=[lat_col, lng_col])

    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(
        plot_nodes[lng_col],
        plot_nodes[lat_col],
        c=plot_nodes["target_total_trips"],
        cmap="viridis",
        s=35,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title("Station trip intensity in space")
    ax.set_xlabel(lng_col)
    ax.set_ylabel(lat_col)
    plt.colorbar(sc, ax=ax, label="incoming + outgoing trips")
    ax.grid(alpha=0.2)
    plt.show()
else:
    print("No latitude/longitude columns found. Available columns:")
    print(list(nodes.columns))

## 9. Interpretation checklist

- If GraphSAGE or GCN beats Random Forest and XGBoost, the station graph carries signal that is missing from ordinary node features.
- If GraphSAGE beats the shuffled-edge GraphSAGE model, the real spatial or mobility connections matter, not just extra neural network capacity.
- If feature-only models win, inspect the features: latitude, distance-to-center, weather, and POI distances may already encode most spatial structure.
- If all models are weak, repeat the experiment across many daily-period graphs and use temporal train/test splits instead of one snapshot.

Exercise extension: aggregate several Warsaw graph files into one training set and test whether GNN gains are larger during morning and afternoon peak periods.